# Simple GNN training example

This notebook trains the repository's actual edge-feature GNN on one prepared bootstrap graph. Before running it, use `gufu.py` cells 1–4 in `RUN_MODE = "stepwise"` to prepare the scans, train or load the AE, and build a matching `GraphState`. Then point `graph_path` below at that graph.

The graph and GNN architecture come from the shared modules so this uses the same inputs and model as GUFU. The optimizer loop and Equation (4) loss are written plainly in this notebook. Nothing is saved to disk.

The paper defines the loss as a sum over physical and virtual edges. There is no term over non-edges and no negative sampling. The code reports both that summed loss and its per-edge mean; only the sum is the objective used for training.

In [1]:
from pathlib import Path
import torch
import torch.nn.functional as F

from graph_build import load_graph_state
from sage_ve import EdgeFeatureSAGE, GNNModelConfig, graph_state_to_tensors

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This is the current gufu.py real-data run. Change it to the graph_state.json
# produced by cells 1–4 if you used a different AE/configuration/run directory.
graph_path = Path(
    "output/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-d256-z64-drop0p1/graph_state.json"
)
if not graph_path.is_file():
    raise FileNotFoundError(
        f"Missing {graph_path}. Run gufu.py cells 1–4 in stepwise mode, then set graph_path to that run's graph_state.json."
    )

graph = load_graph_state(graph_path)
if graph.batch_role != "bootstrap_reference":
    raise ValueError("This example trains only on the labeled bootstrap graph.")
graph_tensors = graph_state_to_tensors(graph, device=device, dtype=torch.float32)
node_count = graph_tensors.node_features.shape[0]
edge_count = graph_tensors.edge_features.shape[0]
if edge_count == 0:
    raise ValueError("Equation (4) needs at least one physical or virtual edge.")
assert graph_tensors.node_features.shape == (node_count, graph.node_feature_dim)
assert graph_tensors.edge_features.shape == (edge_count, graph.node_feature_dim)
assert len(graph_tensors.node_ids) == node_count
assert len(graph_tensors.edge_ids) == edge_count

print(f"graph: {graph_path}")
print(f"dataset/batch: {graph.dataset_id}/{graph.batch_id}")
print(f"graph ID: {graph.graph_id}")
print(f"normalization: {graph.normalization}")
print(f"AE checkpoint SHA-256: {graph.encoder_checkpoint_sha256}")
print(f"nodes: {node_count} ({len(graph.sample_node_ids)} samples, {len(graph.ap_node_ids)} APs)")
print(f"edges: {edge_count} ({len(graph.radio_edges)} physical, {len(graph.virtual_edges)} temporary virtual)")
print(f"feature shapes: nodes={tuple(graph_tensors.node_features.shape)}, edges={tuple(graph_tensors.edge_features.shape)}")
print(f"device: {device}")

graph: output/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-d256-z64-drop0p1/graph_state.json
dataset/batch: GF/fingerprint0
graph ID: graph:5399f3dddabea262caa85c493c5d0ade8bab237c0f31d85a49122d3b5e941ca2
normalization: paper_ae
AE checkpoint SHA-256: b7a11bc1558e65dd3f64cedd411497f645794810264b6a208675a985ad040586
nodes: 1065 (392 samples, 673 APs)
edges: 97030 (20474 physical, 76556 temporary virtual)
feature shapes: nodes=(1065, 66), edges=(97030, 66)
device: cuda


## Model

`EdgeFeatureSAGE` implements the repository's Algorithm 1 node and edge feature updates. Each epoch processes the whole graph: node messages depend on neighboring nodes and edge vectors, so this example does not split the graph into independent shuffled rows.

In [2]:
num_layers = 2
dropout_probability = 0.1
initialization_seed = 0

model_config = GNNModelConfig(
    graph_feature_dim=graph.node_feature_dim,
    edge_feature_dim=graph.node_feature_dim,
    node_output_dim=graph.node_feature_dim,
    edge_output_dim=graph.node_feature_dim,
    num_layers=num_layers,
    dropout_probability=dropout_probability,
    initialization_seed=initialization_seed,
)
model = EdgeFeatureSAGE(model_config).to(device)
print(model)
print(f"trainable parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

EdgeFeatureSAGE(
  (W0): ParameterList(
      (0): Parameter containing: [torch.float32 of size 66x132 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 66x132 (cuda:0)]
  )
  (W1): ParameterList(
      (0): Parameter containing: [torch.float32 of size 66x132 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 66x132 (cuda:0)]
  )
  (W2): ParameterList(
      (0): Parameter containing: [torch.float32 of size 66x198 (cuda:0)]
      (1): Parameter containing: [torch.float32 of size 66x198 (cuda:0)]
  )
)
trainable parameters: 60,984


## Training

Equation (4) is `-sum(log(ReLU(z_u dot z_v)))` over each graph edge once. The lower clamp keeps `log(0)` finite, matching the named numerical choice in `sage_ve.py`. Since the objective only reads final node embeddings, the last layer's edge-update matrix `W2` has no path to this loss; its missing gradient is expected. Earlier `W2` layers can affect later node layers.

In [3]:
epochs = 300
learning_rate = 0.01
epsilon = 1e-12
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

left_rows = graph_tensors.edge_left_rows
right_rows = graph_tensors.edge_right_rows

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    output = model(graph_tensors)
    edge_dot_products = (
        output.node_features[left_rows] * output.node_features[right_rows]
    ).sum(dim=1)
    positive_scores = F.relu(edge_dot_products)
    loss = -torch.log(positive_scores.clamp_min(epsilon)).sum()
    loss.backward()

    # Show where the loss has a gradient in each layer. A final W2 layer
    # is expected to have no gradient because Eq. (4) uses node outputs.
    layer_gradients = []
    for layer in range(num_layers):
        for weight_name in ("W0", "W1", "W2"):
            gradient = getattr(model, weight_name)[layer].grad
            if gradient is None:
                layer_gradients.append(f"{weight_name}[{layer}]=none")
            else:
                layer_gradients.append(
                    f"{weight_name}[{layer}]={torch.linalg.vector_norm(gradient).item():.3g}"
                )

    optimizer.step()
    print(
        f"epoch {epoch + 1:3d}/{epochs} | "
        f"Eq4 sum={loss.item():.6f} | "
        f"per-edge mean={loss.item() / edge_count:.6f} | "
        f"mean dot={edge_dot_products.detach().mean().item():.4f} | "
        f"positive pairs={(edge_dot_products.detach() > 0).float().mean().item():.1%} | "
        f"grad norms: {', '.join(layer_gradients)}"
    )

epoch   1/300 | Eq4 sum=28669.107422 | per-edge mean=0.295466 | mean dot=0.7847 | positive pairs=100.0% | grad norms: W0[0]=3.07e+04, W1[0]=5.32e+04, W2[0]=2.64e+04, W0[1]=7.38e+04, W1[1]=8.45e+04, W2[1]=none
epoch   2/300 | Eq4 sum=11829.871094 | per-edge mean=0.121920 | mean dot=0.8879 | positive pairs=100.0% | grad norms: W0[0]=3.31e+03, W1[0]=5.09e+03, W2[0]=2.95e+03, W0[1]=1.52e+04, W1[1]=1.36e+04, W2[1]=none
epoch   3/300 | Eq4 sum=11429.063477 | per-edge mean=0.117789 | mean dot=0.8915 | positive pairs=100.0% | grad norms: W0[0]=1.47e+03, W1[0]=2.16e+03, W2[0]=1.08e+03, W0[1]=8.3e+03, W1[1]=7.67e+03, W2[1]=none
epoch   4/300 | Eq4 sum=11287.406250 | per-edge mean=0.116329 | mean dot=0.8927 | positive pairs=100.0% | grad norms: W0[0]=724, W1[0]=1.28e+03, W2[0]=807, W0[1]=6.3e+03, W1[1]=5.95e+03, W2[1]=none
epoch   5/300 | Eq4 sum=11243.418945 | per-edge mean=0.115876 | mean dot=0.8933 | positive pairs=100.0% | grad norms: W0[0]=451, W1[0]=800, W2[0]=619, W0[1]=5.66e+03, W1[1]=5.5

In [4]:
blah

NameError: name 'blah' is not defined

In [ ]:
epochs = 300
learning_rate = 0.01
epsilon = 1e-12
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

left_rows = graph_tensors.edge_left_rows
right_rows = graph_tensors.edge_right_rows

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    output = model(graph_tensors)
    edge_dot_products = (
        output.node_features[left_rows] * output.node_features[right_rows]
    ).sum(dim=1)
    positive_scores = F.relu(edge_dot_products)
    loss = -torch.log(positive_scores.clamp_min(epsilon)).sum()
    loss.backward()

    # Show where the loss has a gradient in each layer. A final W2 layer
    # is expected to have no gradient because Eq. (4) uses node outputs.
    layer_gradients = []
    for layer in range(num_layers):
        for weight_name in ("W0", "W1", "W2"):
            gradient = getattr(model, weight_name)[layer].grad
            if gradient is None:
                layer_gradients.append(f"{weight_name}[{layer}]=none")
            else:
                layer_gradients.append(
                    f"{weight_name}[{layer}]={torch.linalg.vector_norm(gradient).item():.3g}"
                )

    optimizer.step()
    print(
        f"epoch {epoch + 1:3d}/{epochs} | "
        f"Eq4 sum={loss.item():.6f} | "
        f"per-edge mean={loss.item() / edge_count:.6f} | "
        f"mean dot={edge_dot_products.detach().mean().item():.4f} | "
        f"positive pairs={(edge_dot_products.detach() > 0).float().mean().item():.1%} | "
        f"grad norms: {', '.join(layer_gradients)}"
    )

epoch   1/300 | Eq4 sum=27049.222656 | per-edge mean=0.278772 | mean dot=0.7915 | positive pairs=100.0% | grad norms: W0[0]=2.91e+04, W1[0]=5.37e+04, W2[0]=2.46e+04, W0[1]=8.29e+04, W1[1]=8.82e+04, W2[1]=none
epoch   2/300 | Eq4 sum=11543.642578 | per-edge mean=0.118970 | mean dot=0.8905 | positive pairs=100.0% | grad norms: W0[0]=2.6e+03, W1[0]=3.86e+03, W2[0]=2.22e+03, W0[1]=1.34e+04, W1[1]=1.29e+04, W2[1]=none
epoch   3/300 | Eq4 sum=11300.587891 | per-edge mean=0.116465 | mean dot=0.8926 | positive pairs=100.0% | grad norms: W0[0]=1.38e+03, W1[0]=2.22e+03, W2[0]=1.24e+03, W0[1]=8.33e+03, W1[1]=7.87e+03, W2[1]=none
epoch   4/300 | Eq4 sum=11115.990234 | per-edge mean=0.114562 | mean dot=0.8940 | positive pairs=100.0% | grad norms: W0[0]=704, W1[0]=1.3e+03, W2[0]=823, W0[1]=5.55e+03, W1[1]=6e+03, W2[1]=none
epoch   5/300 | Eq4 sum=11074.482422 | per-edge mean=0.114135 | mean dot=0.8946 | positive pairs=100.0% | grad norms: W0[0]=345, W1[0]=675, W2[0]=533, W0[1]=4.97e+03, W1[1]=5.48e+

In [ ]:
blah

NameError: name 'blah' is not defined

## Final embeddings

This final pass disables dropout and keeps the embeddings in memory, keyed by the graph's stable node IDs. No checkpoint, feature file, or log file is written.

In [ ]:
model.eval()
with torch.no_grad():
    final_output = model(graph_tensors)
    final_edge_dots = (
        final_output.node_features[left_rows] * final_output.node_features[right_rows]
    ).sum(dim=1)
    final_loss = -torch.log(F.relu(final_edge_dots).clamp_min(epsilon)).sum()

node_embeddings = {
    node_id: final_output.node_features[row].cpu()
    for row, node_id in enumerate(graph_tensors.node_ids)
}
assert len(node_embeddings) == node_count
assert all(vector.shape == (graph.node_feature_dim,) for vector in node_embeddings.values())
print(f"eval Eq4 sum={final_loss.item():.6f}")
print(f"eval per-edge mean={final_loss.item() / edge_count:.6f}")
print(f"final node embedding shape=({len(node_embeddings)}, {graph.node_feature_dim})")
for node_id, vector in list(node_embeddings.items())[:3]:
    print(node_id, vector[:6].tolist())